# **Estudo sobre a Localização de Unidades de Atendimento Soroterápico no Estado de São Paulo para Inclusão na Plataforma Onde Tem**

CNPq - PIBITI (processo 183831/2025-0)

**Aluna:** Julia Graziosi Ortiz, BMACC - ICMC/USP

**Orientadora:** Maristela Oliveira dos Santos - SME/ICMC/USP


As fontes de dados utilizados neste notebook estão disponíveis no Google Drive (Iniciação Científica/Relatório Final/Fontes de Dados).

Os arquivos gerados aqui foram armazenados em Iniciação Científica/Relatório Final/Dados Tratados).

In [ ]:
import os
from google.colab import userdata

import pandas as pd
import geopandas as gpd

import numpy as np

## **1. Fontes de Dados**

In [ ]:
!gdown --folder 1I9qQRKnEAn6M9jGgLCXeDKDc9u_pyhdm -O "Dados Brutos"

In [ ]:
os.makedirs("Dados Tratados", exist_ok=True)

### **Estado de São Paulo:** malha municipal e estruturas de saúde

GeoDataFrame final: `municipios`

Leitura

In [ ]:
# Malha Municipal
gdf_ibge = gpd.read_file('Dados Brutos/SP_Municipios_2024/SP_Municipios_2024.shp')

In [ ]:
# Divisão de Saúde Nacional
df_seidigi = pd.read_csv('Dados Brutos/SEIDIGI_SP.csv')

In [ ]:
# Divisão de Saúde Estadual
df_cve = pd.read_csv('Dados Brutos/CVE_SP.csv')

Integração das bases

In [ ]:
municipios = gdf_ibge.copy()

In [ ]:
# Base: Malha Municipal
cols_malha = ['CD_MUN', 'NM_MUN', 'AREA_KM2', 'geometry']
municipios = municipios[cols_malha]

# O padrão dos códigos dos municípios do IBGE possui 7 digitos
# O código verificador final será removido para ser compatível com o arquivo das Regiões
municipios['CD_MUN'] = municipios['CD_MUN'].str[:-1].astype(int)

municipios.rename(
  columns = {
    'CD_MUN': 'Codigo Municipio',
    'NM_MUN': 'Municipio',
    'AREA_KM2' : 'Area km2'
  },
  inplace = True
)

In [ ]:
# Inclusão de Macrorregiões e Regiões de Saúde (Nacional)
cols_regioes = [
  'Codigo Macrorregiao de Saude',
  'Macrorregiao de Saude',
  'Codigo Regiao de Saude',
  'Regiao de Saude',
  'Codigo Municipio',
  'Populacao Estimada IBGE 2022'
]

municipios = municipios.merge(df_seidigi[cols_regioes], on = 'Codigo Municipio')

# População para int
municipios['Populacao Estimada IBGE 2022'] = municipios['Populacao Estimada IBGE 2022'].str.replace('.', '', regex=False).astype(float).astype(np.int64)

In [ ]:
# Inclusão das divisões estaduais de saúde
df_cve.rename(
  columns = {
      'ID_MUNIC' : 'Codigo Municipio',
      'DRS': 'Codigo Departamento Regional de Saude',
      'NM_DRS' : 'Departamento Regional de Saude',
      'GVE' : 'Codigo Grupo de Vigilancia Epidemiologica',
      'NM_GVE' : 'Grupo de Vigilancia Epidemiologica'
  },
  inplace = True
)

# Colunas de interesse da Regionalização Estadual
cols_cve = [
  'Codigo Municipio',
  'Codigo Departamento Regional de Saude',
  'Departamento Regional de Saude',
  'Codigo Grupo de Vigilancia Epidemiologica',
  'Grupo de Vigilancia Epidemiologica'
]

municipios = municipios.merge(df_cve[cols_cve], on = 'Codigo Municipio' )

**Geocodificação** ($\pm$ 15 minutos)

In [ ]:
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
from tqdm import tqdm       # barra de progresso

In [ ]:
# Inicializa geocodificador
geolocator = Nominatim(user_agent="coordenadas_municipais")
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1)
tqdm.pandas()

# Aplica no GeoDataFrame
municipios['Nominatim'] = municipios['Municipio'].progress_apply(
    lambda x: geocode({
        "city": x,
        "state": "São Paulo",
        "country": "Brasil"
    })
)

In [ ]:
# Adiciona colunas com as coordenadas
municipios['Longitude'] = municipios['Nominatim'].apply(lambda p: p.longitude if p else None)
municipios['Latitude'] = municipios['Nominatim'].apply(lambda p: p.latitude if p else None)
municipios['coordenadas'] = gpd.points_from_xy(municipios['Longitude'], municipios['Latitude'], crs='EPSG:4326')

# Verifica se as coordenadas estão dentro das fronteiras do município
# Garante que a coluna 'wgs84' contenha as geometrias reprojetadas dos municípios
municipios['wgs84'] = municipios.geometry.to_crs(epsg=4326)

# Define uma função para verificar e atualizar as coordenadas
def adjust_point_to_polygon(row):
    point = row['coordenadas']
    polygon = row['wgs84']

    # Verifica se tanto o ponto quanto o polígono são válidos e existem
    if point and polygon and polygon.is_valid and point.is_valid:
        if not polygon.contains(point):
            # Se o ponto estiver fora do polígono, usa o ponto representativo do polígono
            return polygon.representative_point()
        else:
            # Se o ponto estiver dentro, mantém o original
            return point
    else:
        # Se o ponto ou polígono for inválido ou ausente, tenta usar o ponto representativo se o polígono for válido
        if polygon and polygon.is_valid:
            return polygon.representative_point()
        else:
            # Caso contrário, mantém o ponto original (que pode ser None)
            return point

# Aplica a função ao GeoDataFrame para ajustar as coordenadas
municipios['coordenadas'] = municipios.apply(adjust_point_to_polygon, axis=1)

# Atualiza as colunas de Longitude e Latitude com base nas coordenadas potencialmente novas
municipios['Longitude'] = municipios['coordenadas'].apply(lambda p: p.x if p else None)
municipios['Latitude'] = municipios['coordenadas'].apply(lambda p: p.y if p else None)


In [ ]:
# municipios['coordenadas'] = municipios.representative_point()
# municipios['Longitude'] = municipios['coordenadas'].x
# municipios['Latitude'] = municipios['coordenadas'].y

**Trocando as coordenadas dos municípios que tiveram erros ao calcular a distância na API**

Obtidas manualmente pelo Google Maps

In [ ]:
# municipios.loc[municipios['Municipio'] == 'Adamantina', ['Longitude', 'Latitude']] = [-45.215155,-22.831054]
# municipios.loc[municipios['Municipio'] == 'Teodoro Sampaio', ['Longitude', 'Latitude']] = [-52.156879,-22.524786]

Ajustes finais e salvamento

In [ ]:
# Reordenas as colunas
ordem_cols = [
  'Codigo Municipio', 'Municipio',
  'Codigo Macrorregiao de Saude', 'Macrorregiao de Saude',
  'Codigo Regiao de Saude', 'Regiao de Saude',
  'Codigo Departamento Regional de Saude', 'Departamento Regional de Saude',
  'Codigo Grupo de Vigilancia Epidemiologica', 'Grupo de Vigilancia Epidemiologica',
  'Area km2', 'Populacao Estimada IBGE 2022',
  'Latitude', 'Longitude', 'geometry'
]

municipios = municipios[ordem_cols].copy()

In [ ]:
municipios.to_file("Dados Tratados/Estruturas_Saude_SP.gpkg", driver = "GPKG")

In [ ]:
# # Adiciona as coordenadas como geometria secundária
# municipios['coords'] = gpd.points_from_xy(municipios['Longitude'], municipios['Latitude'], crs = 4326)

### **Acidentes Peçonhentos:** registros do SINAN

DataFrame final: `sinan`

In [ ]:
# Registros do SINAN entre 2019 e 2025
df_sinan = pd.read_csv('Dados Brutos/ANIMBR19A25.csv', encoding='latin1')

In [ ]:
# Filtragem de registros atendidos E ocorridos no estado de SP
ambos = (df_sinan['SG_UF_NOT'] == 35) & (df_sinan['ANT_UF'] == 35)
sinan = df_sinan[ambos].copy()

# Verifica se os municípios registrados existem
cd_mun_sp = municipios['Codigo Municipio'].to_list()
origem_valida = sinan['ANT_MUNIC_'].isin(cd_mun_sp)
destino_valido = sinan['ID_MUNICIP'].isin(cd_mun_sp)
sinan = sinan[origem_valida & destino_valido]

In [ ]:
# Variáveis de interesse
colunas = [
  'NU_ANO', 'ID_MUNICIP', 'ANT_MUNIC_', 'ANT_TEMPO_', 'TP_ACIDENT',
  'TRA_CLASSI', 'CON_SOROTE', 'EVOLUCAO',
  'ANI_SERPEN', 'ANI_ARANHA','ANI_LAGART',
  'NU_AMPOLAS', 'NU_AMPOL_1', 'NU_AMPO_5', 'NU_AMPOL_6', 'NU_AMPOL_4',
  'NU_AMPOL_8', 'NU_AMPO_7', 'NU_AMPOL_3', 'NU_AMPOL_9'
]

sinan = sinan[colunas].copy()
sinan.reset_index(drop = True, inplace = True)

Tradução das informações

In [ ]:
sinan.rename(
  columns = {
    'NU_ANO' : 'Ano',
    'ID_MUNICIP' : 'Destino',
    'ANT_MUNIC_' : 'Origem',
    'ANT_TEMPO_' : 'Tempo',
    'TP_ACIDENT' : 'Acidente',
    'TRA_CLASSI' : 'Gravidade',
    'CON_SOROTE' : 'Soroterapia',
    'EVOLUCAO' : 'Evolucao',
    'ANI_SERPEN' : 'Serpente',
    'ANI_ARANHA' : 'Aranha',
    'ANI_LAGART' : 'Lagarta',
    'NU_AMPOLAS' : 'Ampolas SABr' ,  # Soro AntiBotrópico (serpente)
    'NU_AMPOL_1' : 'Ampolas SACr',   # Soro AntiCrotálico (serpente)
    'NU_AMPO_5' : 'Ampolas SABC',   # Soro AntiBotrópico-Crotálico (serpente)
    'NU_AMPOL_6' : 'Ampolas SABL', # Soro AntiBotrópico-Laquético (serpente)
    'NU_AMPOL_4' : 'Ampolas SAEla',  # Soro AntiElapídico (serpente)
    'NU_AMPOL_8' : 'Ampolas SAAr',  # Soro AntiAracnídico ou Antifonêutrico (aranha)
    'NU_AMPO_7' : 'Ampolas SALox',  # Soro AntiLoxoscélico (aranha)
    'NU_AMPOL_3' : 'Ampolas SALon', # Soro AntiLonômico (lagarta)
    'NU_AMPOL_9' : 'Ampolas SAEsc' # Soro AntiEscorpiônico (escorpiao)
  },
  inplace = True
)

In [ ]:
dict_tempos = {1: 'até 1h', 2: '1-3 h', 3: '3-6 h', 4: '6-12 h', 5: '12-24 h', 6: '24+ h', 9: 'Ignorado'}
dict_acidentes = {1: 'Serpente', 2: 'Aranha', 3: 'Escorpião', 4: 'Lagarta', 5: 'Abelha', 6: 'Outros', 9: 'Não informado'}
dict_gravidade = {1: 'Leve', 2: 'Moderado', 3: 'Grave', 9: 'Ignorado'}
dict_soroterapia = {1: 'Sim', 2: 'Não', 9: 'Ignorado'}
dict_evolucao = {1: 'Cura', 2: 'Óbito pelo acidente', 3: 'Óbito por outra causa', 9: 'Ignorado'}
dict_serpentes = {1: 'Botrópico', 2: 'Crotálico', 3: 'Elapídico', 4: 'Laquético', 5: 'Não peçonhento', 9: 'Ignorado'}
dict_aranhas = {1: 'Fonêutrico', 2: 'Loxoscélico', 3: 'Latrodéctico', 4: 'Outra', 9: 'Ignorado'}
dict_lagartas = {1: 'Lonômico', 2: 'Outra', 9: 'Ignorado'}

mapeamentos = [
    ('Tempo', dict_tempos),
    ('Acidente', dict_acidentes),
    ('Gravidade', dict_gravidade),
    ('Soroterapia', dict_soroterapia),
    ('Evolucao', dict_evolucao),
    ('Serpente', dict_serpentes),
    ('Aranha', dict_aranhas),
    ('Lagarta', dict_lagartas)
]

for col, dicionario in mapeamentos:
    sinan[col] = sinan[col].map(dicionario)

In [ ]:
sinan['Origem'] = sinan['Origem'].astype(int)

cols_ampolas_soros = ['Ampolas SABr', 'Ampolas SACr', 'Ampolas SABC', 'Ampolas SABL',
                'Ampolas SAEla', 'Ampolas SAAr', 'Ampolas SALox', 'Ampolas SALon',
                'Ampolas SAEsc']

for col in cols_ampolas_soros:
  sinan[col] = sinan[col].fillna(0).astype(int)

In [ ]:
# Cria uma nova coluna para acidentes com Escorpião
sinan['Escorpião'] = sinan['Acidente'].apply(lambda x: 'Escorpiônico' if x == 'Escorpião' else np.nan)

# Une as colunas das espécies
def especie_acidente(row):
  animal = row['Acidente']
  if animal in ['Aranha', 'Escorpião', 'Lagarta', 'Serpente']:
    return row[animal]
  else:
    return np.nan

sinan['Especie'] = sinan.apply(especie_acidente, axis = 1)
sinan.drop(columns = ['Aranha', 'Escorpião', 'Lagarta', 'Serpente'], inplace = True)

In [ ]:
# Reordena as colunas
ordem_colunas = [
  'Ano', 'Origem', 'Destino', 'Tempo',
  'Acidente', 'Especie', 'Gravidade',
  'Soroterapia', 'Evolucao'
]
ordem_colunas += cols_ampolas_soros
sinan = sinan[ordem_colunas]

In [ ]:
sinan.to_csv("Dados Tratados/SINAN_SP_19A25.csv")

In [ ]:
sinan

### **Unidades de Atendimento Soroterápico**: Pontos Estratégicos para Soro Antiveneno (PESA) disponibilizados pelo CIEVS

GeoDataFrame final: `postos`

In [ ]:
df_pesas = pd.read_csv('Dados Brutos/Unidades_de_Atendimento.csv')

In [ ]:
# Ajuste manual de uma unidade com erro no cadastro
nome_unidade = 'HOSPITAL DE BASE DE SAO JOSE DOS CAMPOS'
municipio_unidade = 'SAO JOSE DOS CAMPOS'

# Remove do df hospitais
filtro = (df_pesas['Municipio'] == municipio_unidade) & (df_pesas['Nome Posto'] == nome_unidade)
indice_unidade = df_pesas[filtro].index

df_pesas.loc[indice_unidade, 'Nome Posto'] = 'DR RUBENS SAVASTANO HOSPITAL REGIONAL DE SAO JOSE DOS CAMPOS'
df_pesas.loc[indice_unidade, 'Endereço'] = 'R. Goiânia, 345 - Parque Industrial, São José dos Campos - SP, 12235-625'
df_pesas.loc[indice_unidade, 'Latitude'] = -23.243041
df_pesas.loc[indice_unidade, 'Longitude'] = -45.906007

In [ ]:
postos = gpd.GeoDataFrame(
  df_pesas,
  geometry = gpd.points_from_xy(df_pesas['Longitude'], df_pesas['Latitude']),
  crs = 4326
)

In [ ]:
# Padroniza o nome do município e adiciona informações
mun_idx = dict(zip(municipios['Codigo Municipio'], municipios.index))
mun_reg = dict(zip(municipios['Codigo Municipio'], municipios['Codigo Regiao de Saude']))
mun_nom = dict(zip(municipios['Codigo Municipio'], municipios['Municipio']))

postos['Municipio'] = postos['Codigo Municipio'].map(mun_nom)
postos['Indice Municipio'] = postos['Codigo Municipio'].map(mun_idx)
postos['Codigo Regiao de Saude'] = postos['Codigo Municipio'].map(mun_reg)

In [ ]:
# Colunas para cada soro
colunas_soros = ['Loxoscélico', 'Fonêutrico', 'Botrópico', 'Crotálico',
                 'Elapídico', 'Laquético', 'Escorpiônico', 'Lonômico']

def quebra_soros_disponíveis(row):
  string = str(row['Tipo Soro']) if pd.notna(row['Tipo Soro']) else ""

  # Possui algum soro
  row['Tem Soro'] = 1 if string else 0

  # Quais soros
  for tipo in colunas_soros:
    if tipo in string:
      row[tipo] = 1
  return row

postos['Tem Soro'] = np.nan
postos[colunas_soros] = np.nan
postos = postos.apply(quebra_soros_disponíveis, axis = 1)

In [ ]:
# Reordena as colunas
ordem_colunas = [
    'Codigo Municipio', 'Municipio', 'Indice Municipio',
    'Codigo Regiao de Saude', 'GVE',
    'Nome Posto', 'Endereço', 'Telefone', '24H',
    'Tem Soro', 'Tipo Soro', 'Acidentes'
]

ordem_colunas += colunas_soros
ordem_colunas += ['Latitude', 'Longitude', 'geometry']

postos = postos[ordem_colunas].copy()

In [ ]:
postos.to_file('Dados Tratados/PESAs_SP.gpkg', driver = 'GPKG')

In [ ]:
postos_ativos = postos[postos['Tem Soro'] == 1]
len(postos_ativos)

In [ ]:
postos_ativos.to_file('Dados Tratados/PESAs_Ativas_SP.gpkg', driver = 'GPKG')

## **2. Cálculo das distâncias entre municípios e PESAs**

In [ ]:
# Verifica se todos os postos tem Nome diferente
print(len(postos))
print(len(postos['Nome Posto'].unique()))

In [ ]:
len(municipios) , len(postos)

In [ ]:
COORD_POSTOS = postos[['Longitude', 'Latitude']].values.tolist()
COORD_POSTOS_ATIVOS = postos_ativos[['Longitude', 'Latitude']].values.tolist()
COORD_MUNICIPIOS = municipios[['Longitude', 'Latitude']].values.tolist()

**API `openrouteservice`**

In [ ]:
!pip install openrouteservice

In [ ]:
import openrouteservice
import time

In [ ]:
def realiza_chamada_api(locais, API_KEY):
  # Inicializa o cliente
  client = openrouteservice.Client(key = API_KEY, base_url = 'https://api.heigit.org/openrouteservice')

  # Locais = lista de coordenadas no formato [longitude, latitude]
  n = len(locais)         # (n-1) pontos + 1 posto
  destino = [n - 1]       # coordenadas no último item da lista
  origens = [i for i in range(n-1)] # indice dos pontos

  # Requisição da matriz de tempo de viagem
  matriz = client.distance_matrix(
      profile = 'driving-car',
      locations = locais,
      sources = origens,
      destinations = destino,
      metrics = ['duration']   # segundos
  )

  tempos = matriz['durations'] # type = list

  return tempos

In [ ]:
dists = pd.DataFrame(np.nan, index = municipios['Municipio'], columns = postos['Nome Posto'])
# dists_ativos = pd.DataFrame(np.nan, index = municipios['Municipio'], columns = postos_ativos['Nome Posto'])

In [ ]:
def calcula_distancias(df_distancias, df_municipios, df_postos, tipo_posto = 'todos', aux = 1):

  # Chaves da API - limite de 500 chamadas por chave
  CHAVE_UM = userdata.get('API_CHAVE1')
  CHAVE_DOIS = userdata.get('API_CHAVE2')

  # Nome dos Municípios = índice
  NOME_MUNICIPIOS = df_municipios['Municipio'].values.tolist()
  COORD_MUNICIPIOS = df_municipios[['Longitude', 'Latitude']].values.tolist()

  # Nome dos Postos = colunas
  NOME_POSTOS = df_postos['Nome Posto'].values.tolist()
  COORD_POSTOS = df_postos[['Longitude', 'Latitude']].values.tolist()
  P = len(NOME_POSTOS)

  if tipo_posto == 'todos':
    postos_usados = 'todos os PESAs'
  else:
    postos_usados = 'os PESAs ativos'
  print(f'Iniciando as chamadas da API para {postos_usados}...')

  ERROS = []
  for p, posto in enumerate(NOME_POSTOS):
    print(f'\n  Posto {p+aux}/{P+(aux-1)}: {posto}')
    coordenadas = COORD_MUNICIPIOS + [COORD_POSTOS[p]]

    chave = CHAVE_UM if p <400 else CHAVE_DOIS
    tempos_api = realiza_chamada_api(coordenadas, chave)
    tempos_api = [t[0] for t in tempos_api]

    erros = []
    for m, municipio in enumerate(NOME_MUNICIPIOS):
      if tempos_api[m] is not None:
        df_distancias.loc[municipio, posto] = tempos_api[m]
      else:
        df_distancias.loc[municipio, posto] = -1
        erros.append(municipio)

    # Exibe municípios que ficaram sem tempo até o posto
    if erros:
      print(f'   * {len(erros)} sem distância: {erros}.')
    else:
      print(f'   * Sucesso! Todas as distâncias foram calculadas.')

    ERROS.append(erros)
    df_distancias.to_csv(f'distancias_{tipo_posto}_segundos.csv')
    time.sleep(5)

  print('\nFinalizadas as chamadas da API.')

  return df_distancias

In [ ]:
distancias = calcula_distancias(dists, municipios, postos, tipo_posto = 'todos', aux = 1)

In [ ]:
municipios['coordenadas'] = gpd.points_from_xy(municipios['Longitude'], municipios['Latitude'])

In [ ]:
distancias_ativos = calcula_distancias(dists_ativos, municipios, postos_ativos, tipo_posto = 'ativos', aux = 1)

In [ ]:
# Erro: Posto 10/242: UNIDADE DE RETAGUARDA DO MELHADO
postos_restantes = postos_ativos.iloc[10:]
distancias_ativos = calcula_distancias(dists_ativos, municipios, postos_restantes, tipo_posto = 'ativos', aux = 11)

In [ ]:
# Erro: Posto 81/241: PRONTO ATENDIMENTO IGARACU DO TIETE
postos_restantes = postos_ativos.iloc[81:]
distancias_ativos = calcula_distancias(dists_ativos, municipios, postos_restantes, tipo_posto = 'ativos', aux = 82)

Erro para o município de Aparecida.

Recalculando as distâncias entre esse município e todos os postos

In [ ]:
def refaz_chamada_api(locais, API_KEY):
  # Inicializa o cliente
  client = openrouteservice.Client(key = API_KEY, base_url = 'https://api.heigit.org/openrouteservice')

  # Locais = lista de coordenadas no formato [longitude, latitude]
  n = len(locais)                         # (n-1) postos + 1 municipio
  destinos = [i for i in range(n-1)]       # postos
  origem = [n - 1]                       # municipio no final

  # Requisição da matriz de tempo de viagem
  matriz = client.distance_matrix(
      profile = 'driving-car',
      locations = locais,
      sources = origem,
      destinations = destinos,
      metrics = ['duration']   # segundos
  )

  tempos = matriz['durations'][0] # type = list

  return tempos

In [ ]:
NOME_POSTOS = postos_ativos['Nome Posto'].values.tolist()
COORD_POSTOS = postos_ativos[['Longitude', 'Latitude']].values.tolist()
REFAZER_MUNICIPIOS = ['Aparecida']
COORD_REFAZER = [[-45.215155,-22.831054]]

M = len(REFAZER_MUNICIPIOS)

CHAVE_USP = 'eyJvcmciOiI1YjNjZTM1OTc4NTExMTAwMDFjZjYyNDgiLCJpZCI6IjRiMGYxOGEzNTliYTQ0MGRhNjNlYTc4Yjk4ZGUwNThlIiwiaCI6Im11cm11cjY0In0='
CHAVE_PESSOAL = 'eyJvcmciOiI1YjNjZTM1OTc4NTExMTAwMDFjZjYyNDgiLCJpZCI6IjgwYjRlYzcyZDU2NzQ5Y2Y4MzdiNThhZGYyYmE5ZGE5IiwiaCI6Im11cm11cjY0In0='

print('Iniciando as chamadas da API...')
inicio_chamadas = time.time()

for m, municipio in enumerate(REFAZER_MUNICIPIOS):
  print(f'\n  Município {m+1}/{M}: {municipio}')
  coordenadas = COORD_POSTOS + [COORD_REFAZER[m]]

  chave = CHAVE_PESSOAL if m <499 else CHAVE_USP
  tempos_api = refaz_chamada_api(coordenadas, chave)
  #tempos_api = [t[0] for t in tempos_api]

  erros = []
  for p, posto in enumerate(NOME_POSTOS):
    if tempos_api[p] is not None:
      distancias_ativos.loc[municipio, posto] = tempos_api[p]
    else:
      distancias_ativos.loc[municipio, posto] = -1
      erros.append(posto)

  # Exibe municípios que ficaram sem tempo até o posto
  if erros:
    print(f'   * {len(erros)} sem distância: {erros}.')
  else:
    print(f'   * Sucesso! Todas as distâncias foram calculadas.')

  distancias_ativos.to_csv('distancias_ativos_aparecida_segundos.csv')
  time.sleep(2)

print('\n\nFinalizadas as chamadas da API.')

##### Recalculando distâncias quando ocorre algum erro

In [ ]:
NOME_POSTOS = postos['Nome Posto'].values.tolist()
NOME_MUNICIPIOS = municipios['Municipio'].values.tolist()
P = len(NOME_POSTOS)

CHAVE_USP = 'eyJvcmciOiI1YjNjZTM1OTc4NTExMTAwMDFjZjYyNDgiLCJpZCI6IjRiMGYxOGEzNTliYTQ0MGRhNjNlYTc4Yjk4ZGUwNThlIiwiaCI6Im11cm11cjY0In0='
CHAVE_PESSOAL = 'eyJvcmciOiI1YjNjZTM1OTc4NTExMTAwMDFjZjYyNDgiLCJpZCI6IjgwYjRlYzcyZDU2NzQ5Y2Y4MzdiNThhZGYyYmE5ZGE5IiwiaCI6Im11cm11cjY0In0='

print('Iniciando as chamadas da API...')
inicio_chamadas = time.time()

for p, posto in enumerate(NOME_POSTOS):
  print(f'\n  Posto {p+1}/{P}: {posto}')
  coordenadas = COORD_MUNICIPIOS + [COORD_POSTOS[p]]

  chave = CHAVE_PESSOAL if p <499 else CHAVE_USP
  tempos_api = realiza_chamada_api(coordenadas, chave)
  tempos_api = [t[0] for t in tempos_api]

  erros = []
  for m, municipio in enumerate(NOME_MUNICIPIOS):
    if tempos_api[m] is not None:
      dists.loc[municipio, posto] = tempos_api[m]
    else:
      dists.loc[municipio, posto] = -1
      erros.append(municipio)

  # Exibe municípios que ficaram sem tempo até o posto
  if erros:
    print(f'   * {len(erros)} sem distância: {erros}.')
  else:
    print(f'   * Sucesso! Todas as distâncias foram calculadas.')

  dists.to_csv('distancias_segundos.csv')
  time.sleep(2)

fim_chamadas = time.time()
duracao_chamadas = fim_chamadas - inicio_chamadas
print(f'\n\nFinalizadas as chamadas da API.\n  Aproximadamente {np.ceil(duracao_chamadas / 60)} minutos.')

In [ ]:
# Fiz besteira e esqueci de colocar duas chaves, parou em
# Posto 500/667 - CS DE RIFAINA
# Posto 624/668: PRONTO ATENDIMENTO DE TAQUARIVAI

NOME_POSTOS_RESTANTES = NOME_POSTOS[623:]
COORD_POSTOS_RESTANTES = COORD_POSTOS[623:]
NOME_MUNICIPIOS = municipios['Municipio'].values.tolist()
P = len(NOME_POSTOS_RESTANTES)

CHAVE_USP = 'eyJvcmciOiI1YjNjZTM1OTc4NTExMTAwMDFjZjYyNDgiLCJpZCI6IjRiMGYxOGEzNTliYTQ0MGRhNjNlYTc4Yjk4ZGUwNThlIiwiaCI6Im11cm11cjY0In0='
CHAVE_PESSOAL = 'eyJvcmciOiI1YjNjZTM1OTc4NTExMTAwMDFjZjYyNDgiLCJpZCI6IjgwYjRlYzcyZDU2NzQ5Y2Y4MzdiNThhZGYyYmE5ZGE5IiwiaCI6Im11cm11cjY0In0='

print('Iniciando as chamadas da API...')
inicio_chamadas = time.time()

for p, posto in enumerate(NOME_POSTOS_RESTANTES):
  print(f'\n  Posto {p+624}/{623+P}: {posto}')
  coordenadas = COORD_MUNICIPIOS + [COORD_POSTOS_RESTANTES[p]]

  chave = CHAVE_PESSOAL if p <499 else CHAVE_USP
  tempos_api = realiza_chamada_api(coordenadas, chave)
  tempos_api = [t[0] for t in tempos_api]

  erros = []
  for m, municipio in enumerate(NOME_MUNICIPIOS):
    if tempos_api[m] is not None:
      dists.loc[municipio, posto] = tempos_api[m]
    else:
      dists.loc[municipio, posto] = -1
      erros.append(municipio)

  # Exibe municípios que ficaram sem tempo até o posto
  if erros:
    print(f'   * {len(erros)} sem distância: {erros}.')
  else:
    print(f'   * Sucesso! Todas as distâncias foram calculadas.')

  dists.to_csv('distancias_segundos.csv')
  time.sleep(2)

fim_chamadas = time.time()
duracao_chamadas = fim_chamadas - inicio_chamadas
print(f'\n\nFinalizadas as chamadas da API.\n  Aproximadamente {np.ceil(duracao_chamadas / 60)} minutos.')

In [ ]:
dists.to_csv('dois_municipios_sem_tempo.csv')

Coordenadas via Google Maps

In [ ]:
nova_coord_aparecida = [-45.229553,-22.847281]
nova_coord_teod_samp = [-52.156879,-22.524786]

In [ ]:
def refaz_chamada_api(locais, API_KEY):
  # Inicializa o cliente
  client = openrouteservice.Client(key = API_KEY, base_url = 'https://api.heigit.org/openrouteservice')

  # Locais = lista de coordenadas no formato [longitude, latitude]
  n = len(locais)                         # (n-1) postos + 1 municipio
  destinos = [i for i in range(n-1)]       # postos
  origem = [n - 1]                       # municipio no final

  # Requisição da matriz de tempo de viagem
  matriz = client.distance_matrix(
      profile = 'driving-car',
      locations = locais,
      sources = origem,
      destinations = destinos,
      metrics = ['duration']   # segundos
  )

  tempos = matriz['durations'][0] # type = list

  return tempos

In [ ]:
len(tempos_api)

In [ ]:
NOME_POSTOS = postos['Nome Posto'].values.tolist()
REFAZER_MUNICIPIOS = ['Aparecida', 'Teodoro Sampaio']
COORD_REFAZER = [[-45.229553,-22.847281], [-52.156879,-22.524786]]
M = len(REFAZER_MUNICIPIOS)

CHAVE_USP = 'eyJvcmciOiI1YjNjZTM1OTc4NTExMTAwMDFjZjYyNDgiLCJpZCI6IjRiMGYxOGEzNTliYTQ0MGRhNjNlYTc4Yjk4ZGUwNThlIiwiaCI6Im11cm11cjY0In0='
CHAVE_PESSOAL = 'eyJvcmciOiI1YjNjZTM1OTc4NTExMTAwMDFjZjYyNDgiLCJpZCI6IjgwYjRlYzcyZDU2NzQ5Y2Y4MzdiNThhZGYyYmE5ZGE5IiwiaCI6Im11cm11cjY0In0='

print('Iniciando as chamadas da API...')
inicio_chamadas = time.time()

for m, municipio in enumerate(REFAZER_MUNICIPIOS):
  print(f'\n  Município {m+1}/{M}: {municipio}')
  coordenadas = COORD_POSTOS + [COORD_REFAZER[m]]

  chave = CHAVE_PESSOAL if m <499 else CHAVE_USP
  tempos_api = refaz_chamada_api(coordenadas, chave)
  #tempos_api = [t[0] for t in tempos_api]

  erros = []
  for p, posto in enumerate(NOME_POSTOS):
    if tempos_api[p] is not None:
      dists.loc[municipio, posto] = tempos_api[p]
    else:
      dists.loc[municipio, posto] = -1
      erros.append(posto)

  # Exibe municípios que ficaram sem tempo até o posto
  if erros:
    print(f'   * {len(erros)} sem distância: {erros}.')
  else:
    print(f'   * Sucesso! Todas as distâncias foram calculadas.')

  dists.to_csv('distancias_segundos.csv')
  time.sleep(2)

fim_chamadas = time.time()
duracao_chamadas = fim_chamadas - inicio_chamadas
print(f'\n\nFinalizadas as chamadas da API.\n  Aproximadamente {np.ceil(duracao_chamadas / 60)} minutos.')

In [ ]:
dists.to_csv("Dados Tratados/Distancias_API_Municipios_Postos.csv")

In [ ]:
dists